In [52]:
import ast
import nbformat
from pathlib import Path
import pandas as pd
def cargar_funciones(ruta_nb, namespace, incluir_constantes=True):
    nb = nbformat.read(ruta_nb, as_version=4)
    for celda in nb.cells:
        if celda.cell_type != "code":
            continue
        # ast no entiende magics (%, !)
        fuente = "\n".join(l for l in celda.source.splitlines()
                           if not l.lstrip().startswith(("%", "!")))
        try:
            arbol = ast.parse(fuente)
        except SyntaxError:
            continue

        nodos = []
        for n in arbol.body:
            if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef,
                              ast.AsyncFunctionDef, ast.ClassDef)):
                nodos.append(n)
            elif (incluir_constantes and isinstance(n, ast.Assign)
                  and all(isinstance(t, ast.Name) and t.id.isupper()
                          for t in n.targets)):
                try:                      # solo constantes literales
                    ast.literal_eval(n.value)
                    nodos.append(n)
                except ValueError:
                    pass

        if nodos:
            modulo = ast.Module(body=nodos, type_ignores=[])
            exec(compile(modulo, ruta_nb, "exec"), namespace)

cargar_funciones("./01_scraping_coruna.ipynb", globals())

RAW_DIR = Path("../datos/raw/edicion_anterior")
INTERIM_DIR = Path("../datos/interim/edicion_anterior")
for d in (RAW_DIR, INTERIM_DIR):
    d.mkdir(parents=True, exist_ok=True)

EDICION_ANTERIOR = {
    "anio": 2025,
    "num_edicion": 58,          # <- rellenar con el NN de la URL
    "fecha_consulta": "2026-11-18",
}
assert EDICION_ANTERIOR["num_edicion"] is not None, "Falta el número de edición"

In [68]:
EXCLUIR = ["PITUFOS", "BENXAMIN", "ALEVIN", "INFANTIL", "CADETE", "XUVENIL"]

def descubrir_carreras_absoluta(num_edicion, prefijo="ant2025_carrera"):
    html = descargar_pagina(
        f"{BASE_URL}/es/web/resultado/edicion-{num_edicion}/eventos",
        f"eventos_edicion_{num_edicion}_historico.html"
    )

    soup = BeautifulSoup(html, "lxml")
    eventos = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "/es/web/resultado/evento-" not in href:
            continue

        id_ev = href.rstrip("/").split("evento-")[-1]
        nombre = a.get_text(" ", strip=True)

        if any(e["id_evento"] == id_ev for e in eventos):
            continue

        html_ev = descargar_pagina(
            f"{BASE_URL}{href}",
            f"evento_{id_ev}_2025_resultados.html"
        )

        soup_ev = BeautifulSoup(html_ev, "lxml")

        url_abs = None

        for enlace in soup_ev.find_all("a", href=True):
            texto = enlace.get_text(" ", strip=True).upper()

            if "ABSOLUTA" in texto and not any(
                k in texto for k in EXCLUIR
            ):
                url_abs = enlace["href"]
                break

        if url_abs:
            eventos.append({
                "id_evento": id_ev,
                "nombre": nombre,
                "url_absoluta": url_abs
            })

    for i, evento in enumerate(eventos, start=1):
        evento["id_carrera"] = f"{prefijo}_{i:02d}"

    return eventos


def extraer_carrera(id_carrera, url_absoluta, tam_pagina=25):
    filas_carrera, previos, pagina, paginas_ok = [], set(), 1, 0
    while True:
        nombre_raw = f"{id_carrera}_pag_{pagina:02d}.html"
        html = descargar_pagina(f"{BASE_URL}{url_absoluta}?page={pagina}", nombre_raw)
        filas = parsear_pagina_resultados(html, id_carrera, pagina)

        if not filas:
            (RAW_DIR / nombre_raw).unlink(missing_ok=True)
            break
        actuales = {f["dorsal"] for f in filas}
        if actuales == previos:
            (RAW_DIR / nombre_raw).unlink(missing_ok=True)
            break

        previos = actuales
        filas_carrera.extend(filas)
        paginas_ok += 1
        if len(filas) < tam_pagina:
            break
        pagina += 1
    return pd.DataFrame(filas_carrera), paginas_ok

In [54]:
resultados_edicion_anterior = []
registro_extraccion = []

for carrera in carreras_prev:
    id_carrera = carrera["id_carrera"]

    print(f"Procesando {id_carrera}: {carrera['nombre']}")

    df_carrera, n_pag = extraer_carrera(
        id_carrera,
        carrera["url_absoluta"]
    )

    df_carrera.to_csv(
        INTERIM_DIR / f"{id_carrera}_interim.csv",
        index=False,
        encoding="utf-8"
    )

    resultados_edicion_anterior.append(df_carrera)

    duplicados = df_carrera.duplicated(subset=["dorsal"]).sum()

    registro_extraccion.append({
        "id_carrera": id_carrera,
        "evento_id": carrera["id_evento"],
        "nombre": carrera["nombre"],
        "paginas": n_pag,
        "participantes": len(df_carrera),
        "duplicados_dorsal": duplicados,
        "estado": "OK" if len(df_carrera) > 0 and duplicados == 0 else "REVISAR"
    })

    print(f"  {len(df_carrera)} participantes en {n_pag} páginas")


df_anterior = pd.concat(resultados_edicion_anterior, ignore_index=True)

df_registro = pd.DataFrame(registro_extraccion)

df_anterior.to_csv(
    INTERIM_DIR / "resultados_edicion_2025.csv",
    index=False,
    encoding="utf-8"
)

df_registro.to_csv(
    INTERIM_DIR / "registro_extraccion.csv",
    index=False,
    encoding="utf-8"
)

print("\nExtracción terminada")
print("Total de participantes:", len(df_anterior))
print("Total de carreras:", len(carreras_prev))
print("CSV guardado en:", INTERIM_DIR.resolve())

Procesando ant_carrera_01: RESULTADOSACTUALEVENTOSTorre de Hercules
  [CACHÉ] ant_carrera_01_pag_01.html
  [CACHÉ] ant_carrera_01_pag_02.html
  [CACHÉ] ant_carrera_01_pag_03.html
  [CACHÉ] ant_carrera_01_pag_04.html
  [CACHÉ] ant_carrera_01_pag_05.html
  [CACHÉ] ant_carrera_01_pag_06.html
  [CACHÉ] ant_carrera_01_pag_07.html
  [CACHÉ] ant_carrera_01_pag_08.html
  [CACHÉ] ant_carrera_01_pag_09.html
  [CACHÉ] ant_carrera_01_pag_10.html
  [CACHÉ] ant_carrera_01_pag_11.html
  [CACHÉ] ant_carrera_01_pag_12.html
  [CACHÉ] ant_carrera_01_pag_13.html
  [CACHÉ] ant_carrera_01_pag_14.html
  [CACHÉ] ant_carrera_01_pag_15.html
  [CACHÉ] ant_carrera_01_pag_16.html
  [CACHÉ] ant_carrera_01_pag_17.html
  [CACHÉ] ant_carrera_01_pag_18.html
  [CACHÉ] ant_carrera_01_pag_19.html
  [CACHÉ] ant_carrera_01_pag_20.html
  [CACHÉ] ant_carrera_01_pag_21.html
  [CACHÉ] ant_carrera_01_pag_22.html
  [CACHÉ] ant_carrera_01_pag_23.html
  [CACHÉ] ant_carrera_01_pag_24.html
  [CACHÉ] ant_carrera_01_pag_25.html
  [CACH

In [55]:
c = carreras_prev[0]
df, n_pag = extraer_carrera(c["id_carrera"], c["url_absoluta"])
print(c["id_carrera"], len(df), "filas en", n_pag, "páginas")
print(sorted(df.columns))   # compara con las columnas de 2026
display(df[["id_carrera", "pagina_origen", "puesto", "puesto_sexo",
            "puesto_categoria", "tiempo_oficial"]].head(5))

  [CACHÉ] ant_carrera_01_pag_01.html
  [CACHÉ] ant_carrera_01_pag_02.html
  [CACHÉ] ant_carrera_01_pag_03.html
  [CACHÉ] ant_carrera_01_pag_04.html
  [CACHÉ] ant_carrera_01_pag_05.html
  [CACHÉ] ant_carrera_01_pag_06.html
  [CACHÉ] ant_carrera_01_pag_07.html
  [CACHÉ] ant_carrera_01_pag_08.html
  [CACHÉ] ant_carrera_01_pag_09.html
  [CACHÉ] ant_carrera_01_pag_10.html
  [CACHÉ] ant_carrera_01_pag_11.html
  [CACHÉ] ant_carrera_01_pag_12.html
  [CACHÉ] ant_carrera_01_pag_13.html
  [CACHÉ] ant_carrera_01_pag_14.html
  [CACHÉ] ant_carrera_01_pag_15.html
  [CACHÉ] ant_carrera_01_pag_16.html
  [CACHÉ] ant_carrera_01_pag_17.html
  [CACHÉ] ant_carrera_01_pag_18.html
  [CACHÉ] ant_carrera_01_pag_19.html
  [CACHÉ] ant_carrera_01_pag_20.html
  [CACHÉ] ant_carrera_01_pag_21.html
  [CACHÉ] ant_carrera_01_pag_22.html
  [CACHÉ] ant_carrera_01_pag_23.html
  [CACHÉ] ant_carrera_01_pag_24.html
  [CACHÉ] ant_carrera_01_pag_25.html
  [CACHÉ] ant_carrera_01_pag_26.html
  [CACHÉ] ant_carrera_01_pag_27.html
 

,id_carrera,pagina_origen,puesto,puesto_sexo,puesto_categoria,tiempo_oficial
0,ant_carrera_01,1,1,M-1,SNRM-1,00:20:30
1,ant_carrera_01,1,2,M-2,SNRM-2,00:21:00
2,ant_carrera_01,1,3,M-3,SUB23M-1,00:21:01
3,ant_carrera_01,1,4,M-4,SNRM-3,00:21:19
4,ant_carrera_01,1,5,M-5,MS1M-1,00:21:26


In [56]:
print("RAW_DIR     :", RAW_DIR.resolve())
print("INTERIM_DIR :", INTERIM_DIR.resolve())

for nombre, d in [("raw edición anterior", RAW_DIR),
                  ("interim edición anterior", INTERIM_DIR),
                  ("raw general", Path("../datos/raw")),
                  ("interim general", Path("../datos/interim"))]:
    ficheros = sorted(p.name for p in d.glob("*")) if d.exists() else None
    print(f"\n{nombre}: ", "NO EXISTE" if ficheros is None else f"{len(ficheros)} ficheros")
    if ficheros:
        print(ficheros[:8], "..." if len(ficheros) > 8 else "")

RAW_DIR     : /Users/davidmanuelmarinmartinez/Documents/Classes/sistemasBD/p2-coruna-corre/datos/raw/edicion_anterior
INTERIM_DIR : /Users/davidmanuelmarinmartinez/Documents/Classes/sistemasBD/p2-coruna-corre/datos/interim/edicion_anterior

raw edición anterior:  226 ficheros
['ant_carrera_01_pag_01.html', 'ant_carrera_01_pag_02.html', 'ant_carrera_01_pag_03.html', 'ant_carrera_01_pag_04.html', 'ant_carrera_01_pag_05.html', 'ant_carrera_01_pag_06.html', 'ant_carrera_01_pag_07.html', 'ant_carrera_01_pag_08.html'] ...

interim edición anterior:  8 ficheros
['ant_carrera_01_interim.csv', 'ant_carrera_02_interim.csv', 'ant_carrera_03_interim.csv', 'ant_carrera_04_interim.csv', 'ant_carrera_05_interim.csv', 'ant_carrera_06_interim.csv', 'registro_extraccion.csv', 'resultados_edicion_2025.csv'] 

raw general:  322 ficheros
['carrera_01_pag_01.html', 'carrera_01_pag_02.html', 'carrera_01_pag_03.html', 'carrera_01_pag_04.html', 'carrera_01_pag_05.html', 'carrera_01_pag_06.html', 'carrera_01_pa

In [66]:
url = "https://corunacorre.com/es/web/resultado/edicion-58/eventos"

html = descargar_pagina(url, "eventos_2025_correctos.html")
soup = BeautifulSoup(html, "lxml")

for a in soup.find_all("a", href=True):
    href = a["href"]
    texto = a.get_text(" ", strip=True)

    if "evento-" in href:
        print(texto, "->", href)

  [DESCARGA] https://corunacorre.com/es/web/resultado/edicion-58/eventos
XIII CARREIRA POPULAR TORRE DE HÉRCULES -TROFEO J.A. VARELA CALVIÑO V ANDAINA SOLIDARIA -> /es/web/resultado/evento-2938
IX CARREIRA POPULAR MATOGRANDE - XUXÁN -> /es/web/resultado/evento-2943
XVI BAIXADA POPULAR DE CROSS SAN PEDRO DE VISMA -> /es/web/resultado/evento-2944
XII CARREIRA POPULAR OS ROSALES -> /es/web/resultado/evento-2945
XII CARREIRA POPULAR VOLTA A OZA -> /es/web/resultado/evento-2946
XI CARREIRA POPULAR VENTORRILLO -> /es/web/resultado/evento-2947
XIII CARREIRA POPULAR NOVO MESOIRO -> /es/web/resultado/evento-2948


In [67]:
html = descargar_pagina(
    f"{BASE_URL}/es/web/resultado/evento-2938",
    "evento_2938_2025.html"
)

soup = BeautifulSoup(html, "lxml")

print("Título:", soup.title.get_text(" ", strip=True))

for a in soup.find_all("a", href=True):
    texto = a.get_text(" ", strip=True).upper()
    if "ABSOLUTA" in texto:
        print(texto, "->", a["href"])

  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2938
Título: Resultados de
    Torre de Hercules
ABSOLUTA (2007 E ANTERIORES) -> /es/web/resultado/competicion-15696


In [69]:
carreras_prev = descubrir_carreras_absoluta(
    EDICION_ANTERIOR["num_edicion"]
)

for c in carreras_prev:
    print(
        c["id_carrera"],
        "|",
        c["id_evento"],
        "|",
        c["nombre"],
        "|",
        c["url_absoluta"]
    )

print("\nTotal de carreras:", len(carreras_prev))

  [DESCARGA] https://corunacorre.com/es/web/resultado/edicion-58/eventos
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2938
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2943
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2944
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2945
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2946
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2947
  [DESCARGA] https://corunacorre.com/es/web/resultado/evento-2948
ant2025_carrera_01 | 2938 | XIII CARREIRA POPULAR TORRE DE HÉRCULES -TROFEO J.A. VARELA CALVIÑO V ANDAINA SOLIDARIA | /es/web/resultado/competicion-15696
ant2025_carrera_02 | 2943 | IX CARREIRA POPULAR MATOGRANDE - XUXÁN | /es/web/resultado/competicion-15705
ant2025_carrera_03 | 2944 | XVI BAIXADA POPULAR DE CROSS SAN PEDRO DE VISMA | /es/web/resultado/competicion-15706
ant2025_carrera_04 | 2945 | XII CARREIRA POPULAR OS ROSALES | /es/web/resultado/competicion-

In [70]:
resultados_edicion_anterior = []
registro_extraccion = []

for carrera in carreras_prev:
    id_carrera = carrera["id_carrera"]

    print(f"\nProcesando {id_carrera}: {carrera['nombre']}")

    df_carrera, n_pag = extraer_carrera(
        id_carrera,
        carrera["url_absoluta"]
    )

    df_carrera.to_csv(
        INTERIM_DIR / f"{id_carrera}_interim.csv",
        index=False,
        encoding="utf-8"
    )

    resultados_edicion_anterior.append(df_carrera)

    registro_extraccion.append({
        "id_carrera": id_carrera,
        "evento_id": carrera["id_evento"],
        "nombre": carrera["nombre"],
        "paginas": n_pag,
        "participantes": len(df_carrera),
        "duplicados_dorsal": df_carrera.duplicated(
            subset=["dorsal"]
        ).sum(),
        "estado": "OK" if len(df_carrera) > 0 else "REVISAR"
    })

    print(f"{len(df_carrera)} participantes en {n_pag} páginas")

df_anterior = pd.concat(
    resultados_edicion_anterior,
    ignore_index=True
)

df_anterior.to_csv(
    INTERIM_DIR / "resultados_edicion_2025.csv",
    index=False,
    encoding="utf-8"
)

df_registro = pd.DataFrame(registro_extraccion)

df_registro.to_csv(
    INTERIM_DIR / "registro_extraccion.csv",
    index=False,
    encoding="utf-8"
)

display(df_registro)

print("\nTotal participaciones:", len(df_anterior))
print("Total carreras:", df_anterior["id_carrera"].nunique())


Procesando ant2025_carrera_01: XIII CARREIRA POPULAR TORRE DE HÉRCULES -TROFEO J.A. VARELA CALVIÑO V ANDAINA SOLIDARIA
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=1
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=2
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=3
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=4
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=5
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=6
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=7
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=8
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=9
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=10
  [DESCARGA] https://corunacorre.com/es/web/resultado/competicion-15696?page=11
  [DESCAR

,id_carrera,evento_id,nombre,paginas,participantes,duplicados_dorsal,estado
0,ant2025_carrera_01,2938,XIII CARREIRA POPULAR TORRE DE HÉRCULES -TROFE...,63,1562,0,OK
1,ant2025_carrera_02,2943,IX CARREIRA POPULAR MATOGRANDE - XUXÁN,38,930,0,OK
2,ant2025_carrera_03,2944,XVI BAIXADA POPULAR DE CROSS SAN PEDRO DE VISMA,26,646,0,OK
3,ant2025_carrera_04,2945,XII CARREIRA POPULAR OS ROSALES,32,798,0,OK
4,ant2025_carrera_05,2946,XII CARREIRA POPULAR VOLTA A OZA,27,653,0,OK
5,ant2025_carrera_06,2947,XI CARREIRA POPULAR VENTORRILLO,29,701,0,OK
6,ant2025_carrera_07,2948,XIII CARREIRA POPULAR NOVO MESOIRO,23,567,0,OK



Total participaciones: 5857
Total carreras: 7


In [77]:
df_anterior_limpio = df_anterior.copy()

df_anterior_limpio["sexo"] = (
    df_anterior_limpio["puesto_sexo"]
    .str.extract(r"^([MF])-\d+$")[0]
)

df_anterior_limpio["categoria"] = (
    df_anterior_limpio["puesto_categoria"]
    .str.extract(r"^(.+)-\d+$")[0]
)

df_anterior_limpio["tiempo_segundos"] = (
    pd.to_timedelta(df_anterior_limpio["tiempo_oficial"])
    .dt.total_seconds()
    .astype("int64")
)

display(
    df_anterior_limpio[
        ["puesto_sexo", "sexo", "puesto_categoria",
         "categoria", "tiempo_oficial", "tiempo_segundos"]
    ].head(10)
)

,puesto_sexo,sexo,puesto_categoria,categoria,tiempo_oficial,tiempo_segundos
0,M-1,M,SUB23M-1,SUB23M,00:20:26,1226
1,M-2,M,MS1M-1,MS1M,00:21:02,1262
2,M-3,M,SNRM-1,SNRM,00:21:03,1263
3,M-4,M,MS1M-2,MS1M,00:21:30,1290
4,M-5,M,SNRM-2,SNRM,00:21:52,1312
5,M-6,M,MS1M-3,MS1M,00:21:55,1315
6,M-7,M,SNRM-3,SNRM,00:22:04,1324
7,M-8,M,MS1M-4,MS1M,00:22:05,1325
8,M-9,M,MS2M-1,MS2M,00:22:15,1335
9,M-10,M,SUB20M-1,SUB20M,00:22:33,1353


In [78]:
print("Participaciones:", len(df_anterior_limpio))

print("\nNulos en las nuevas columnas:")
print(
    df_anterior_limpio[
        ["sexo", "categoria", "tiempo_segundos"]
    ].isna().sum()
)

print("\nDistribución por sexo:")
print(df_anterior_limpio["sexo"].value_counts(dropna=False))

print("\nDistribución por categoría:")
print(df_anterior_limpio["categoria"].value_counts(dropna=False))

Participaciones: 5857

Nulos en las nuevas columnas:
sexo               24
categoria           0
tiempo_segundos     0
dtype: int64

Distribución por sexo:
sexo
M      3878
F      1955
NaN      24
Name: count, dtype: int64

Distribución por categoría:
categoria
SNRM      1111
MS1M      1009
MS2M       964
SNRF       863
MS3M       462
MS1F       432
MS2F       391
MS4M       158
MS3F       120
SUB23M     114
SUB23F      92
SUB20M      58
MS4F        29
SUB20F      25
NO          24
ESPF         3
ESPM         2
Name: count, dtype: int64


In [79]:

sin_sexo = df_anterior_limpio[
    df_anterior_limpio["sexo"].isna()
]

display(
    sin_sexo[
        ["id_carrera", "puesto", "dorsal", "nombre_completo",
         "puesto_sexo", "puesto_categoria", "tiempo_oficial"]
    ]
)

,id_carrera,puesto,dorsal,nombre_completo,puesto_sexo,puesto_categoria,tiempo_oficial
722,ant2025_carrera_01,723,699,Sin Datos del Dorsal —,—,NO-1,00:35:24
892,ant2025_carrera_01,893,3625,Sin Datos del Dorsal —,—,NO-2,00:37:04
1166,ant2025_carrera_01,1167,727,Sin Datos del Dorsal —,—,NO-3,00:40:21
1561,ant2025_carrera_01,1563,1670,Sin Datos del Dorsal —,—,NO-4,01:01:30
1771,ant2025_carrera_02,210,1672,Sin Datos del Dorsal —,—,NO-1,00:30:30
1785,ant2025_carrera_02,224,158,Sin Datos del Dorsal —,—,NO-2,00:30:40
2306,ant2025_carrera_02,745,727,Sin Datos del Dorsal —,—,NO-3,00:40:11
2581,ant2025_carrera_03,90,235,Sin Datos del Dorsal —,—,NO-1,00:29:23
2732,ant2025_carrera_03,241,570,Sin Datos del Dorsal —,—,NO-2,00:33:51
2797,ant2025_carrera_03,306,83,Sin Datos del Dorsal —,—,NO-3,00:35:04


In [ ]:
# Las participaciones sin sexo identificable se mantienen como NA.
df_anterior_limpio["sexo"] = df_anterior_limpio["sexo"].astype("string")

# Comprobación
print("Total participaciones:", len(df_anterior_limpio))
print("Sexo desconocido:", df_anterior_limpio["sexo"].isna().sum())
print("\nDistribución por sexo:")
print(df_anterior_limpio["sexo"].value_counts(dropna=False))